In [103]:
import numpy as np


In [104]:
gridWorld=np.zeros((2,5,5), dtype=float)

lake=(0,0)
fire=[(4,4)]
smoke=[(1,2),(3,2)]
boulders=[(3,4),(2,4)]

print(gridWorld)

[[[0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]]]


In [105]:
def terminal(state):
    if tuple(state[1:3]) in fire and state[0]:
        return True
    elif tuple(state[1:3]) in boulders:
        return True 
    else:
        return False
    
def reward(state):
    pos = tuple(state[1:3])
    if pos in fire and state[0]:
        return 100
    elif pos in smoke:
        return -10
    elif pos in boulders:
        return -100
    else:
        return -1

def movement(state, dirc):
    if dirc=='N':
        if tuple(state[1:3])==(0,1) and not state[0]:
            return (1, 0,0)
        return state if state[2] == 0 else (state[0], state[1], state[2]-1)
    
    elif dirc=='S':
        return state if state[2] == 4 else (state[0], state[1], state[2]+1)
    
    elif dirc=='E':
        return state if state[1] == 4 else (state[0], state[1]+1, state[2])
    
    elif dirc=='W':
        if tuple(state[1:3])==(1,0) and not state[0]:
            return (1, 0,0)
        return state if state[1] == 0 else (state[0], state[1]-1, state[2])
    
    else:
        return state
    
def possible_states(state,dirc):
    if dirc == 'N':
        return [movement(state, 'N'), movement(state, 'E'), movement(state, 'W'),movement(state, '#')]
    elif dirc == 'S':
        return [movement(state, 'S'), movement(state, 'E'), movement(state, 'W'),movement(state, '#')]
    elif dirc == 'E':
        return [movement(state, 'E'), movement(state, 'N'), movement(state, 'S'),movement(state, '#')]
    elif dirc == 'W':
        return [movement(state, 'W'), movement(state, 'N'), movement(state, 'S'),movement(state, '#')]
    

In [106]:
LOWER_BOUND=-1000
def value_func( state, gamma):
    value=LOWER_BOUND
    if tuple(state[1:3]) in smoke:
        for dirc in ['N','S','E','W']:
            temp_val=0
            p_states = possible_states(state, dirc)
            temp_val += 0.4 * (reward(p_states[0]) + gamma * gridWorld[p_states[0][0], p_states[0][1], p_states[0][2]])
            temp_val += 0.4 * (reward(p_states[3]) + gamma * gridWorld[p_states[3][0], p_states[3][1], p_states[3][2]])
            for i in range(1,3):
                p_state = p_states[i]
                temp_val += 0.1 * (reward(p_state) + gamma * gridWorld[p_state[0], p_state[1], p_state[2]])
            value = max(value, temp_val)
                    
    else:
        for dirc in ['N','S','E','W']:
            temp_val=0
            p_states = possible_states(state, dirc)
            temp_val += 0.7 * (reward(p_states[0]) + gamma * gridWorld[p_states[0][0], p_states[0][1], p_states[0][2]])
            for i in range(1,4):
                p_state = p_states[i]
                temp_val += 0.1 * (reward(p_state) + gamma * gridWorld[p_state[0], p_state[1], p_state[2]])
            value = max(value, temp_val)
        
    return value


In [107]:
def policy_iteration(gamma, theta):
    for _ in range(10):
        delta = 0
        for i in range(1,-1,-1):
            for j in range(4,-1,-1):
                for k in range(4,-1,-1):
                    state = [i,j,k]
                    if not terminal(state):
                        v = gridWorld[i,j,k]
                        gridWorld[i,j,k] = value_func(state, gamma)
                        delta = max(delta, abs(v - gridWorld[i,j,k]))
                    


In [108]:
policy_iteration(0.9, 0.01)
gridWorld

array([[[ 15.05874634,  18.58729004,  11.98714221,   7.81197645,
           4.01108652],
        [ 18.58729004,  14.44272543,   2.30119712,   2.88737247,
           0.82004742],
        [ 13.95918536,  10.22646019,   3.05233437,  -1.04972552,
           0.        ],
        [  9.70407905,   6.15381857,  -5.27103882, -13.613755  ,
           0.        ],
        [  5.64165456,   2.40474393,  -2.67621374,  -6.08504832,
          -6.59487658]],

       [[ 23.76776566,  23.44340198,  17.4689292 ,  16.16231423,
          12.05990568],
        [ 29.21235338,  29.59970698,  17.82966203,  20.78540873,
           9.92421246],
        [ 35.41429873,  39.0839363 ,  32.11422441,  29.12222492,
           0.        ],
        [ 42.12761679,  48.42227194,  48.25811015,  54.7999356 ,
           0.        ],
        [ 49.52827096,  60.07026592,  72.88298506,  91.00987613,
           0.        ]]])